# FLRRC v1.4 training (TinyLlama 1.1B + LoRA, Colab T4)

Run cells top to bottom. **Runtime -> Change runtime type -> T4 GPU first.**
Changes vs v1.3: 75 reviewed samples, answer ends with `</s>`, distinct pad token,
learning rate 2e-4 and ~95 steps (v1.3 ran only 39 steps at the default LR, loss 2.54 -> 2.34).

In [1]:
BRANCH = "feature/react-frontend-v2.0"   # change to "dev" once the dataset PR is merged

!git clone https://github.com/AdhimulamBhargavSaiViswanath-05/flrrc-decision-engine.git
%cd flrrc-decision-engine
!git checkout {BRANCH}
!git pull origin {BRANCH}
!wc -l data/training/chatml-v1.4.jsonl

Cloning into 'flrrc-decision-engine'...
remote: Enumerating objects: 207, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 207 (delta 7), reused 47 (delta 5), pack-reused 145 (from 1)
Receiving objects: 100% (207/207), 1.15 MiB | 999.00 KiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/flrrc-decision-engine
branch 'feature/react-frontend-v2.0' set up to track 'origin/feature/react-frontend-v2.0'.
Switched to a new branch 'feature/react-frontend-v2.0'
From https://github.com/AdhimulamBhargavSaiViswanath-05/flrrc-decision-engine
 * branch            feature/react-frontend-v2.0 -> FETCH_HEAD
Already up to date.
75 data/training/chatml-v1.4.jsonl


The line above must print **75**. If the file is missing, the dataset commit is not pushed to this branch yet.

In [2]:
!pip install -q -U transformers peft accelerate trl datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 19.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


If the next cell shows an import error, use **Runtime -> Restart session** and run again from this cell. Do NOT downgrade trl/transformers.

In [3]:
import inspect, torch, transformers, peft, trl
from trl import SFTTrainer, SFTConfig

print("GPU:", torch.cuda.get_device_name(0))
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__)

cfg_params = inspect.signature(SFTConfig.__init__).parameters
trn_params = inspect.signature(SFTTrainer.__init__).parameters
print("SFTConfig has max_length:", "max_length" in cfg_params, "| max_seq_length:", "max_seq_length" in cfg_params)
print("SFTTrainer has processing_class:", "processing_class" in trn_params, "| tokenizer:", "tokenizer" in trn_params)

GPU: Tesla T4
transformers 5.20.2 | peft 0.21.2 | trl 1.16.0
SFTConfig has max_length: True | max_seq_length: False
SFTTrainer has processing_class: True | tokenizer: False


In [4]:
# T4 fix used in v1.3 (only exists in newer trl, so it is guarded)
try:
    import trl.kernels.chunked_logprob as _k
    def _patched_addmm(acc, a, b):
        acc.addmm_(a.float(), b.float())
    _k._addmm_fp32 = _patched_addmm
    print("Patched T4 kernel")
except Exception as e:
    print("No kernel patch needed:", type(e).__name__)

Patched T4 kernel


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.unk_token   # NOT eos, so the </s> label is never masked as padding
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj","v_proj","k_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="data/training/chatml-v1.4.jsonl")["train"]
print("samples:", len(dataset))
SYSTEM_PROMPT = dataset[0]["messages"][0]["content"]
print("system prompt:", SYSTEM_PROMPT)

def fmt(ex):
    # Same layout as v1.3 (<|role|>\ncontent\n), plus </s> after the assistant answer so the model learns to stop.
    txt = ""
    for m in ex["messages"]:
        if m["role"] == "assistant":
            txt += f"<|assistant|>\n{m['content']}</s>"
        else:
            txt += f"<|{m['role']}|>\n{m['content']}\n"
    return txt

sample_txt = fmt(dataset[0])
ids = tokenizer(sample_txt).input_ids
print("tokens in sample 0:", len(ids))
print("last 6 tokens:", tokenizer.convert_ids_to_tokens(ids[-6:]))
print("max tokens over dataset:", max(len(tokenizer(fmt(e)).input_ids) for e in dataset))

Generating train split: 0 examples [00:00, ? examples/s]

samples: 75
system prompt: You are FLRRC Decision Engine. Given a Question, output 5 boxes as JSON: box_1_flrrc, box_2_equation, box_3_graph, box_4_risk, box_5_next_action.
tokens in sample 0: 547
last 6 tokens: ['▁che', 'ppi', '▁half', '▁day', '"}', '</s>']
max tokens over dataset: 886


Check the output above: last tokens should end with the JSON closing `}` then `</s>` (a single `</s>`). Max tokens must be below 1024.

In [12]:
from transformers import TrainingArguments
from trl import SFTConfig

# Separate standard TrainingArguments from SFT-specific arguments
training_args_dict = dict(
    output_dir="models/flrrc-v1.4-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    packing=False,
    fp16=False,
    bf16=False,
)

# Configure SFTConfig with only its supported properties
sft_args = {}
if "max_length" in cfg_params:
    sft_args["max_length"] = 1024
elif "max_seq_length" in cfg_params:
    sft_args["max_seq_length"] = 1024

# Create the final configuration using SFTConfig but passing our training arguments
config = SFTConfig(
    **sft_args,
    output_dir=training_args_dict.pop("output_dir")
)

# Apply the remaining TrainingArguments attributes to the config object safely
for key, value in training_args_dict.items():
    setattr(config, key, value)

tok_kw = "processing_class" if "processing_class" in trn_params else "tokenizer"
trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=fmt,
    **{tok_kw: tokenizer},
)
print("Trainer ready | tokenizer kwarg:", tok_kw, "| expected steps ~", (len(dataset) // 4 + 1) * 5)

Applying formatting function to train dataset:   0%|          | 0/75 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/75 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/75 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/75 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/75 [00:00<?, ? examples/s]

Trainer ready | tokenizer kwarg: processing_class | expected steps ~ 95


In [13]:
trainer.train()
for row in trainer.state.log_history:
    if "loss" in row:
        print(f"step {row['step']:>4}  loss {row['loss']:.4f}")

Step,Training Loss
5,2.517966
10,2.149864
15,1.688968
20,1.565026
25,1.304586
30,1.301764
35,1.146658
40,1.195322
45,1.160063
50,1.059854


Step,Training Loss
5,2.517966
10,2.149864
15,1.688968
20,1.565026
25,1.304586
30,1.301764
35,1.146658
40,1.195322
45,1.160063
50,1.059854


step    5  loss 2.5180
step   10  loss 2.1499
step   15  loss 1.6890
step   20  loss 1.5650
step   25  loss 1.3046
step   30  loss 1.3018
step   35  loss 1.1467
step   40  loss 1.1953
step   45  loss 1.1601
step   50  loss 1.0599
step   55  loss 0.9992
step   60  loss 0.9785
step   65  loss 0.9384
step   70  loss 0.9156
step   75  loss 1.0340
step   80  loss 0.9616
step   85  loss 0.9174
step   90  loss 0.9457
step   95  loss 0.9783


Compare with v1.3: loss went 2.54 -> 2.34 in 39 steps. For v1.4 the loss should fall much further. Paste the loss table back before continuing.

In [14]:
trainer.model.save_pretrained("models/flrrc-v1.4-lora-final")
tokenizer.save_pretrained("models/flrrc-v1.4-lora-final")
!ls -lh models/flrrc-v1.4-lora-final/

total 16M
-rw-r--r-- 1 root root 1.2K Oct 20 10:55 adapter_config.json
-rw------- 1 root root  13M Oct 20 10:55 adapter_model.safetensors
-rw-r--r-- 1 root root  410 Oct 20 10:55 chat_template.jinja
-rw-r--r-- 1 root root 5.2K Oct 20 10:55 README.md
-rw-r--r-- 1 root root  399 Oct 20 10:55 tokenizer_config.json
-rw-r--r-- 1 root root 3.5M Oct 20 10:55 tokenizer.json


In [15]:
import json, re

trainer.model.config.use_cache = True
trainer.model.gradient_checkpointing_disable()
trainer.model.eval()

def build_prompt(question):
    return f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\nQuestion: {question}\n<|assistant|>\n"

tests = [
    "Gym ki vellala? (energy 3/10, sleep 5 hrs, exam repu)",
    "I have a job offer at 18 LPA from a startup and my MNC pays 12 LPA. Should I switch?",
    "College ki vellala vadda? (attendance 68%, lab today, energy 4/10)",
    "Should I move to Bangalore?",
]

for q in tests:
    inputs = tokenizer(build_prompt(q), return_tensors="pt").to(trainer.model.device)
    out = trainer.model.generate(**inputs, max_new_tokens=800, do_sample=True, temperature=0.7, top_p=0.9,
                                 repetition_penalty=1.05, eos_token_id=tokenizer.eos_token_id,
                                 pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    try:
        data = json.loads(text)
        ok = all(k in data for k in ["box_1_flrrc","box_2_equation","box_3_graph","box_4_risk","box_5_next_action"])
        status = f"VALID JSON, 5 boxes: {ok}"
    except Exception as e:
        status = f"NOT valid JSON ({type(e).__name__})"
    print("=" * 70)
    print("Q:", q)
    print("->", status)
    print(text[:1200])

[transformers] Both `max_new_tokens` (=800) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=800) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Gym ki vellala? (energy 3/10, sleep 5 hrs, exam repu)
-> VALID JSON, 5 boxes: True
{"box_1_flrrc": {"fact": "Energy 3/10, sleep 5 hrs, exam repu", "logic": "Low energy + low sleep + exam stress adds up", "reasoning": "Rest is the key to focus and performance, so avoid exercise", "representation": "Decision matrix: Focus vs Relaxation vs Energy loss", "conclusion": "Avoid for now, keep study time flexible"}, "box_2_equation": {"formula": "Score = (Focus*0.3)+(Relaxation*0.3)-(Energy_Loss*0.4)", "go_score": "Go (Gym) = (6*0.3)+(7*0.3)-(8*0.4) = 2+2-2.8 = -0.4/10", "nogo_score": "NoGo (Study alone) = (3*0.3)+(4*0.3)-(2*0.4) = 0.9+1.2-0.8 = 1.5/10", "regret_probability": "Exam stress regret 55% (lose sleep), fitness regret 30% (gym chesi energy loss)"}, "box_3_graph": "Bar chart: Go -0.4 vs NoGo 1.5. Rest is better", "box_4_risk": {"best": "Study time flexible, no energy loss", "worst": "Exam stress, gym lo study skip", "middle": "Afternoon study, 3 hrs rest"}, "box_5_next_action": "Tod

[transformers] Both `max_new_tokens` (=800) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: I have a job offer at 18 LPA from a startup and my MNC pays 12 LPA. Should I switch?
-> VALID JSON, 5 boxes: True
{"box_1_flrrc": {"fact": "Job offer 18 LPA start-up, MNC 12 LPA", "logic": "18 LPA + 30% salary cut is 22 LPA loss", "reasoning": "Startups can grow, but the risk is high", "representation": "Bar chart: Risk vs Growth vs Money loss", "conclusion": "Switch only if the growth really is worth it"}, "box_2_equation": {"formula": "Score = (Risk*0.4)+(Growth*0.3)-(Money_Loss*0.3)", "go_score": "Go (Switch) = (7*0.4)+(6*0.3)-(4*0.3) = 2.8+2.1-1.2 = 3.9/10", "nogo_score": "NoGo (Stay) = (2*0.4)+(5*0.3)-(1*0.3) = 0.8+1.5-0.3 = 2.3/10", "regret_probability": "Stay chesi 50% regret (loss), switch chesi 30% regret (growth)"}, "box_3_graph": "Bar chart: Go 3.9 vs NoGo 2.3. Money loss is a big risk", "box_4_risk": {"best": "Startup growth, MNC growth track record", "worst": "Startup closes down, MNC growth is flat", "middle": "Startup culture, MNC structure, pay difference"}, "box_5_n

[transformers] Both `max_new_tokens` (=800) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: College ki vellala vadda? (attendance 68%, lab today, energy 4/10)
-> VALID JSON, 5 boxes: True
{"box_1_flrrc": {"fact": "Attendance 68%, lab today, energy 4/10", "logic": "Attendance 68% + lab = 72% loss", "reasoning": "Energy 4/10 + 4 hrs lab = 8/12 loss", "representation": "Decision matrix: Attendance gain vs Lab time loss vs Learning loss", "conclusion": "Go today, study only"}, "box_2_equation": {"formula": "Score = (Attendance_Gain*0.3)+(Lab_Time_Loss*0.3)-(Learning_Loss*0.4)", "go_score": "Go (Study today) = (9*0.3)+(8*0.3)-(8*0.4) = 2.7+2.1-3.2 = 1.4/10", "nogo_score": "NoGo (Study tomorrow) = (6*0.3)+(3*0.3)-(5*0.4) = 1.8+0.9-1.5 = 1.2/10", "regret_probability": "Today regret 45% (attendance low), tomorrow regret 30% (lab time loss)"}, "box_3_graph": "Bar chart: Go 1.4 vs NoGo 1.2. Learning loss is big", "box_4_risk": {"best": "8 hours study today and attendance 80%", "worst": "Lab today and study today fail", "middle": "Study for 3 hrs today and go to lab"}, "box_5_next_ac

In [16]:
!zip -r flrrc-v1.4-lora.zip models/flrrc-v1.4-lora-final -q
!ls -lh flrrc-v1.4-lora.zip
from google.colab import files
files.download("flrrc-v1.4-lora.zip")

-rw-r--r-- 1 root root 10M Oct 20 10:59 flrrc-v1.4-lora.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>